# ReXKG Pipeline (PyHealth Style)

This notebook shows a PyHealth-native ReXKG workflow similar to other examples:

1. Load dataset
2. Set tasks
3. Build sample datasets
source

predictions_path = Path.cwd() / "result" / "run_relation" / "predictions.json"
if not predictions_path.exists():
    raise FileNotFoundError(f"Missing relation predictions file: {predictions_path}")

structured_output_path = Path.cwd() / "data" / "your_test_file.json"
processed_docs = rexkg_reverse_structure(
    input_json_file=str(predictions_path),
    save_json_file=str(structured_output_path),
)

print("Input predictions:", predictions_path)
print("Structured output:", structured_output_path)
print("Converted documents:", len(processed_docs))

In [1]:
# Make sure to install the needed libraries used for the rexkg PyHealth files. 
# Kernal for this conda virtual environment is running 3.13.13
#!python -m pip install neraug
#!python -m pip install torch

In [ ]:
# may take a few mins to run to build cache
from pathlib import Path
import sys
import importlib.util


# Make sure the local PyHealth package is importable from this notebook.
# Notebook location: PyHealth/examples/rexkg/load_dataset.ipynb
# Package root:      PyHealth/
pyhealth_root = Path.cwd().resolve().parents[1]
if str(pyhealth_root) not in sys.path:
    sys.path.insert(0, str(pyhealth_root))

from pyhealth.datasets import RexKGDataset 
# from pyhealth.models import RexKG
from pyhealth.tasks import (
    RexKGEntityExtractionRadiology,
    RexKGRelationExtractionRadiology,
    RexKGReverseStructureRadiology,
    RexKGGetEntitiesRadiology,
)

/home/strawhat/miniconda3/envs/cs598-pyhealth/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Configure Paths

Set `PROJECT_ROOT` to your repo root if auto-detection does not match your environment.

In [3]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "PyHealth").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

REXKG_DATA_ROOT = PROJECT_ROOT / "src" / "ner" / "data"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("REXKG_DATA_ROOT:", REXKG_DATA_ROOT)
print("Data root exists:", REXKG_DATA_ROOT.exists())

PROJECT_ROOT: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples
REXKG_DATA_ROOT: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/src/ner/data
Data root exists: False


## 2) Load RexKGDataset - Data Preperation

In [4]:
# Prefer repo-relative split files created by src/ner/data/structure_data.py
split_root = PROJECT_ROOT / "PyHealth" / "examples" / "rexkg" / "data" / "data_split"
if not split_root.exists():
    # Fallback when running from PyHealth/examples/rexkg
    split_root = Path.cwd() / "data" / "data_split"

train_json = split_root / "train.json"
test_json = split_root / "test.json"
dev_json = split_root / "test.json"

for p in [train_json, dev_json, test_json]:
    if not p.exists():
        raise FileNotFoundError(f"Missing split file: {p}")

# dataset = RexKGDataset(root=str(expected))
train_dataset = RexKGDataset(root=str(train_json))
dev_dataset = RexKGDataset(root=str(dev_json))
test_dataset = RexKGDataset(root=str(test_json))

## 3) Apply ReXKG Pipeline Tasks 
## Run Entity Pipeline

In [5]:
entity_task = RexKGEntityExtractionRadiology()


# Force output under this notebook folder.
entity_output_dir = Path.cwd() / "result" / "run_entity"
metrics = RexKGEntityExtractionRadiology.set_task(
    train_data=train_dataset,
    dev_data=dev_dataset,
    test_data=test_dataset,
    model="bert-base-uncased",
    output_dir=str(entity_output_dir),
    do_train=True,
    do_eval=True,
    eval_test=True,
    learning_rate=1e-5,
    task_learning_rate=5e-4,
    train_batch_size=8,
    eval_batch_size=64,
    num_epoch=1,
    context_window=5,
)
print(metrics)

pred_file = entity_output_dir / "ent_pred_mimic_headct.json"
print("Expected prediction file:", pred_file)
print("Prediction file exists:", pred_file.exists())

# I think I need to make these set_tasts above instead of run_entity_pipeline
# entity_samples = dataset.set_task(entity_task)





# relation_samples = dataset.set_task(relation_task)
# kg_samples = dataset.set_task(kg_task)

# print("Entity samples:", len(entity_samples))
# print("Relation samples:", len(relation_samples))
# print("KG samples:", len(kg_samples))

Some weights of BertForEntity were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['ner_classifier.0.network.0.bias', 'ner_classifier.0.network.0.weight', 'ner_classifier.0.network.3.bias', 'ner_classifier.0.network.3.weight', 'ner_classifier.1.bias', 'ner_classifier.1.weight', 'width_embedding.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  0%|          | 0/1 [00:00<?, ?it/s]

Evaluating...
Accuracy: 0.973025
Cor: 2685, Pred TOT: 3628, Gold TOT: 3699
P: 0.74008, R: 0.72587, F1: 0.73291
Used time: 3.442178
Saving model to /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity...


100%|██████████| 1/1 [00:48<00:00, 48.19s/it]


Evaluating...
Accuracy: 0.973025
Cor: 2685, Pred TOT: 3628, Gold TOT: 3699
P: 0.74008, R: 0.72587, F1: 0.73291
Used time: 3.742135
Total pred entities: 3628
Output predictions to /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity/ent_pred_mimic_headct.json..
{'output_dir': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity', 'log_file': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity/train.log', 'task': 'mimic01', 'model': 'bert-base-uncased', 'best_dev_f1': 0.7329056912788317, 'test_f1': 0.7329056912788317, 'test_prediction_file': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity/ent_pred_mimic_headct.json'}
Expected prediction file: /mnt/c/Users/Aarje/OneDrive - University of Illinois -

## 4) Run Relation Pipeline

All input/output paths below stay inside `PyHealth/examples/rexkg/`.

In [ ]:
relation_output_dir = Path.cwd() / "result" / "run_relation"

# ner_src_dir points to src/ner so BertForRelation and generate_relation_data can be imported.
# Auto-detection walks up from relation_output_dir; set explicitly if it fails.
NER_SRC_DIR = str(PROJECT_ROOT / "src" / "ner")

relation_metrics = RexKGRelationExtractionRadiology.set_task(
    train_file=str(train_json),
    entity_output_dir=str(entity_output_dir),
    entity_predictions_dev="ent_pred_mimic_headct.json",
    entity_predictions_test="ent_pred_mimic_headct.json",
    model="bert-base-uncased",
    output_dir=str(relation_output_dir),
    do_train=True,
    do_eval=True,
    eval_with_gold=True,
    do_lower_case=True,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=1,
    context_window=20,
    max_seq_length=256,
    ner_src_dir=NER_SRC_DIR,
)
print(relation_metrics)

relation_pred_file = relation_output_dir / "predictions.json"
print("Expected relation prediction file:", relation_pred_file)
print("Relation prediction file exists:", relation_pred_file.exists())

device: cuda, n_gpu: 1
Writing example 0 of 31904
*** Example ***
guid: 0@0::(0,0)-(1,1)
tokens: [CLS] [unused1] unchanged [unused2] [unused3] position [unused4] of the left upper ex ##tre ##mity pic ##c line . [SEP]
input_ids: 101 2 15704 3 4 2597 5 1997 1996 2187 3356 4654 7913 16383 27263 2278 2240 1012 102 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
input_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

Some weights of BertForRelation were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'layer_norm.bias', 'layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Start epoch #0 (lr = 5e-05)...


# Step 5 Reverse Graph Constructed
Build entity/relation tables from reversed JSON

In [ ]:
predictions_path = Path.cwd() / "result" / "run_relation" / "predictions.json"
if not predictions_path.exists():
    raise FileNotFoundError(f"Missing relation predictions file: {predictions_path}")

structured_output_path = Path.cwd() / "data" / "your_test_file.json"
processed_docs = RexKGReverseStructureRadiology.set_task(
    input_json_file=str(predictions_path),
    save_json_file=str(structured_output_path),
)

print("Input predictions:", predictions_path)
print("Structured output:", structured_output_path)
print("Converted documents:", len(processed_docs))

print("\nFirst 10 lines of structured output JSON:")
with structured_output_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if 2231 <= i <= 2282:
            print(f"{i:04d}: {line.rstrip()}")
        if i > 2282:
            break

## Step 5) Get Entities 
data task --- will local smoke be fine???

In [ ]:
# Build entity and relation summary CSV files from Step 5 output JSON.
entity_csv_output_dir = Path.cwd() / "result" / "local_run" / "entities"
relation_csv_output_dir = Path.cwd() / "result" / "local_run" / "relation"

get_entities_result = RexKGGetEntitiesRadiology.set_task(
    ent_pred_mimic_headct=str(structured_output_path),
    ent_real_pred_mimic_headct=str(structured_output_path),
    save_entity_dir=str(entity_csv_output_dir),
    save_real_dir=str(relation_csv_output_dir),
)

print(get_entities_result)
print("Entity CSV folder:", entity_csv_output_dir)
print("Relation CSV folder:", relation_csv_output_dir)
print("all_entities.csv exists:", (entity_csv_output_dir / "all_entities.csv").exists())
print("all_relations.csv exists:", (relation_csv_output_dir / "all_relations.csv").exists())

## Step 6) Get UMLS 
models

In [ ]:
# !python get_umls_entities.py --save_entity_dir ../result/local_run/entities

# !python filter_cui.py --save_entity_dir ../result/local_run/entities

# !python structure_entities.py --save_entity_dir ../result/local_run/entities --ignore_count 1

# !python get_kg_nodes.py \
#   --save_entity_dir ../result/local_run/entities \
#   --save_real_dir ../result/local_run/relation \
#   --save_kg_dir ../result/local_run/kg

# !python get_size_relations.py \
#   --entity_dir ../result/local_run/entities \
#   --real_dir ../result/local_run/relation

# python get_inference_data.py



# !ls -lh /content/drive/MyDrive/cs598_project/src/kg_construct/result/local_run/kg